# Chapter 14 — Calibration

**Book alignment:** Embeddings From First Principles, Chapter 14

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** A cosine of 0.81 is a point on two overlapping bell
curves. Build the positive and hard-negative distributions, read the overlap, and derive an
operating point from false-acceptance / false-rejection rates. On RELATE the
same-claim-vs-different-claim decision from raw cosine has AUC 0.75 and **86%** of pairs in
the escalate band — and the equal-error threshold drifts 0.10 cosine across domains (the
committed Wave 1 measurements).

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. Reproduce: overlapping distributions, FAR, FRR, an ambiguity band

In [ ]:
pos = rng.normal(0.87, 0.06, 2000)
neg = rng.normal(0.77, 0.08, 2000)       # HARD negatives - topic-matched, not random

def far(t): return float((neg >= t).mean())
def frr(t): return float((pos < t).mean())

ts = np.linspace(0.5, 1.0, 501)
eer_t = ts[np.argmin([abs(far(t) - frr(t)) for t in ts])]
t_lo = ts[np.argmin([abs(far(t) - 0.05) for t in ts])]      # 5%-FAR
t_hi = ts[np.argmin([abs(frr(t) - 0.05) for t in ts])]      # 5%-FRR
band = float(((pos >= min(t_lo, t_hi)) & (pos <= max(t_lo, t_hi))).mean()
             + ((neg >= min(t_lo, t_hi)) & (neg <= max(t_lo, t_hi))).mean()) / 2

print(f"equal-error threshold {eer_t:.3f}  (FAR=FRR ~ {far(eer_t):.2f})")
print(f"ambiguity band [{min(t_lo, t_hi):.3f}, {max(t_lo, t_hi):.3f}]  ~ {band:.0%} of pairs -> escalate")
assert far(eer_t) > 0.15                  # overlapping distributions: no clean threshold
print("a bare threshold picks which error to make silently; accept / reject / escalate is honest")

## 2. The measured calibration + threshold drift on RELATE (Wave 1)

In [ ]:
cal = art("wave1", "calibration.json")
dr = art("wave1", "threshold-drift.json")["by_domain"]

print(f"AUC {cal['auc']:.3f}   equal-error {cal['equal_error_rate']['far']:.0%} at t={cal['equal_error_rate']['threshold']:.3f}")
print(f"escalate-band fraction: {cal['escalate_band_fraction']:.0%} of all pairs")
print("\nper-domain equal-error threshold:")
for d, v in sorted(dr.items(), key=lambda kv: kv[1]["eer_threshold"]):
    print(f"  {d:20} {v['eer_threshold']:.3f}")

spread = max(v["eer_threshold"] for v in dr.values()) - min(v["eer_threshold"] for v in dr.values())
assert cal["auc"] < 0.8 and cal["escalate_band_fraction"] > 0.8
assert spread > 0.09                       # ~0.10 - wider than the whole Chapter 11 operating margin
print(f"\nthreshold drift across domains: {spread:.2f}  -  'similarity > 0.8' is not a specification")

## What we earned

A similarity score is only interpretable against the positive and negative distributions it
came from — and with *hard* negatives those distributions overlap: AUC 0.75, and 86% of
pairs land where a threshold cannot decide. The honest output is accept / reject / escalate.
A calibrated threshold is bound to `(model version, corpus snapshot, query type, metric)`
and drifts 0.10 cosine across domains — re-derive it on any change.

**Notebook 15 / Chapter 15** asks whether one scalar was ever the right object, and pulls
several independent geometric signals from the same pair.